<a href="https://colab.research.google.com/github/neuromatch/course-template/blob/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/W1D2_Tutorial5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/neuromatch/course-template/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/W1D2_Tutorial5.ipynb" target="_blank"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

In [ ]:
# @title Download course data
import os, urllib.request

_BASE = "https://raw.githubusercontent.com/neuromatch/course-template/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/"
for _f in ['data/spike_times.npz', 'data/neuron_metadata.npz', 'image/experimental_setup.png']:
    if not os.path.exists(_f):
        os.makedirs(os.path.dirname(_f), exist_ok=True)
        try:
            urllib.request.urlretrieve(_BASE + _f, _f)
        except OSError as _e:
            raise RuntimeError(f"Could not download {_f} ({_e}). On Kaggle, turn Internet on in the notebook settings.") from None
del _f

# Tutorial 5: Working with Data Files

This tutorial demonstrates how to work with data files in your course using the `{nma-data}` directive.

In [ ]:
import sys
if sys.platform == "emscripten":  # only true in JupyterLite/Pyodide
    import micropip
    await micropip.install(["ipywidgets"])

## Declaring Data Files

The `{nma-data}` directive tells the course kit which files your tutorial needs:

This declaration:
- **In the browser**: embeds files ≤2MB into the page (they load instantly from memory)
- **In notebooks**: generates a hidden download cell that fetches them from GitHub

Files must live in `data/` or `image/` inside your day's folder.

## Using Images

Images work the same way as data files - declare them in `{nma-data}` and reference them in your content:

<img src="https://raw.githubusercontent.com/neuromatch/course-template/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/image/experimental_setup.png" alt="Experimental setup schematic" width="80%">

*Our experimental pipeline: stimulus presentation, neural recording, and spike train analysis.*

In the browser, the image loads instantly from the embedded data. In generated notebooks, the download cell fetches it from GitHub.

## Loading Data

Load the data the same way in both environments - normal NumPy works:

In [ ]:
import numpy as np

# Load spike data (allow_pickle=True needed for ragged arrays stored as object dtype)
spike_data = np.load('data/spike_times.npz', allow_pickle=True)
metadata = np.load('data/neuron_metadata.npz', allow_pickle=True)

print(f"Loaded {len(spike_data['neuron_ids'])} neurons")
print(f"Loaded {len(spike_data['trial_ids'])} trials")
print(f"Loaded {len(spike_data['condition_ids'])} conditions")

## Visualizing Spike Trains

Let's create a raster plot showing spike times for the first neuron across all trials:

In [ ]:
import matplotlib.pyplot as plt

# Get spikes for neuron 0 across all trials
n_trials = len(spike_data['trial_ids'])
neuron_idx = 0

fig, ax = plt.subplots(figsize=(10, 6))

for trial in range(n_trials):
    spike_idx = neuron_idx * n_trials + trial
    spikes = spike_data['spikes'][spike_idx]
    ax.scatter(spikes, [trial] * len(spikes), c='black', s=1, marker='|')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Trial')
ax.set_title(f'{metadata["neuron_labels"][neuron_idx]} - Spike Raster')
ax.set_ylim(-1, n_trials)
plt.tight_layout()

## Computing Firing Rates

Calculate mean firing rate by condition:

In [ ]:
n_neurons = len(spike_data['neuron_ids'])
n_conditions = len(spike_data['condition_ids'])

# Compute firing rates
firing_rates = np.zeros((n_neurons, n_conditions))

for neuron in range(n_neurons):
    for trial in range(n_trials):
        spike_idx = neuron * n_trials + trial
        condition = metadata['trial_conditions'][trial]
        n_spikes = len(spike_data['spikes'][spike_idx])
        firing_rates[neuron, condition] += n_spikes / 1.0  # 1 second trials

# Average across trials
firing_rates /= (n_trials // n_conditions)

# Plot firing rates
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(firing_rates, aspect='auto', cmap='viridis')
ax.set_xlabel('Condition')
ax.set_ylabel('Neuron')
ax.set_title('Mean Firing Rate by Condition')
ax.set_xticks(range(n_conditions))
ax.set_xticklabels(metadata['condition_labels'])
plt.colorbar(im, ax=ax, label='Firing Rate (Hz)')
plt.tight_layout()

## Interactive Exploration

Use a widget to explore different neurons and conditions:

In [ ]:
import ipywidgets as widgets

@widgets.interact(
    neuron=widgets.IntSlider(min=0, max=19, value=0, description='Neuron'),
    condition=widgets.IntSlider(min=0, max=4, value=0, description='Condition')
)
def explore_spikes(neuron, condition):
    # Get trials for this condition
    condition_trials = np.where(metadata['trial_conditions'] == condition)[0]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    
    for i, trial in enumerate(condition_trials):
        spike_idx = neuron * n_trials + trial
        spikes = spike_data['spikes'][spike_idx]
        ax.scatter(spikes, [i] * len(spikes), c='black', s=2, marker='|')
    
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Trial (within condition)')
    ax.set_title(f'{metadata["neuron_labels"][neuron]} - {metadata["condition_labels"][condition]}')
    ax.set_ylim(-1, len(condition_trials))
    plt.tight_layout()
    plt.show()

## Summary

When adding data and images to your tutorials:

1. **Create `data/` and/or `image/` folders** in your day's directory
2. **Keep files small** (<2MB total) so they embed in browser pages
3. **Declare all files** with `{nma-data}` at the top of the tutorial
4. **Reference normally** - images with `{figure}`, data with `np.load()` or similar
5. **Validate** with `nma-kit check` before committing

The kit automatically:
- Embeds small files for instant browser loading
- Generates download cells for Colab/Kaggle notebooks
- Validates that your code references only declared files

**Note:** This tutorial includes generation scripts (`generate_data.py`, `generate_schematic.py`) for reproducibility, but you don't need to commit generation scripts in your own courses - just commit the files themselves.